# 10. Deployment Preparation

The saved model is checked in the same way it will be loaded for the Streamlit application.

In [1]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

model_path = Path("../models/loan_recovery_model.pkl")

if not model_path.exists():
    raise FileNotFoundError(
        f"Saved model not found at: {model_path.resolve()}\n"
        "Run 09_model_saving.ipynb successfully before running this notebook."
    )

saved_model = joblib.load(model_path)

required_keys = {"model", "feature_columns"}
missing_keys = required_keys - set(saved_model.keys())
if missing_keys:
    raise KeyError(f"Saved model file is missing required keys: {sorted(missing_keys)}")

model = saved_model["model"]
feature_columns = saved_model["feature_columns"]

if not feature_columns:
    raise ValueError("The saved feature_columns list is empty.")

model_feature_count = getattr(model, "n_features_in_", None)
if model_feature_count is not None and model_feature_count != len(feature_columns):
    raise ValueError(
        f"Feature mismatch: model expects {model_feature_count} features, "
        f"but {len(feature_columns)} feature names were saved."
    )

print("Saved model loaded successfully!")
print("Model type:", type(model).__name__)
print("Number of features:", len(feature_columns))
print("Model name:", saved_model.get("model_name", "Not recorded"))
print("Recall:", saved_model.get("recall", "Not recorded"))
print("ROC-AUC:", saved_model.get("roc_auc", "Not recorded"))
print("Feature names:")
print(feature_columns)

Saved model loaded successfully!
Model type: RandomForestClassifier
Number of features: 21
Model name: Random Forest
Recall: 0.918405
ROC-AUC: 0.913436
Feature names:
['age', 'income', 'credit_score', 'loan_amount', 'loan_term_months', 'interest_rate', 'debt_to_income', 'existing_loans', 'previous_defaults', 'payment_delays', 'employment_years', 'collateral_value', 'contact_success_rate', 'employment_type_Contract', 'employment_type_Salaried', 'employment_type_Self-Employed', 'employment_type_Unemployed', 'loan_purpose_Education', 'loan_purpose_Home', 'loan_purpose_Personal', 'loan_purpose_Vehicle']


## Final Check

In [2]:
processed_path = Path("../data/03_processed_data.pkl")

if not processed_path.exists():
    raise FileNotFoundError(
        f"Processed data not found at: {processed_path.resolve()}"
    )

processed_data = joblib.load(processed_path)
X_test = processed_data["X_test"]

# Ensure the test data contains exactly the features used by the saved model.
missing_features = [col for col in feature_columns if col not in X_test.columns]
extra_features = [col for col in X_test.columns if col not in feature_columns]

if missing_features or extra_features:
    raise ValueError(
        "Feature columns in X_test do not match the saved model.\n"
        f"Missing features: {missing_features}\n"
        f"Unexpected features: {extra_features}"
    )

# Arrange columns in the exact order expected by the model.
X_test = X_test[feature_columns]

if X_test.empty:
    raise ValueError("X_test is empty; cannot run the deployment check.")

sample = X_test.iloc[[0]]
sample_prediction = model.predict(sample)[0]

if not hasattr(model, "predict_proba"):
    raise AttributeError("The saved model does not support predict_proba().")

sample_probability = model.predict_proba(sample)[0, 1]

print("Sample prediction:", int(sample_prediction))
print("Recovery probability:", f"{sample_probability:.4f} ({sample_probability * 100:.2f}%)")
print("Test data shape:", X_test.shape)
print("Deployment check completed successfully!")

Sample prediction: 1
Recovery probability: 0.8477 (84.77%)
Test data shape: (4940, 21)
Deployment check completed successfully!


## Next Step

The model file can now be used by the Streamlit application.